# Bench4BL

In [18]:
import pandas as pd
import os
import json
import re
import xml.etree.ElementTree as ET

In [19]:
# Root directory of the Bench4BL dataset
bench4bl_directory = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/Bench4BL/data/'

### Walks through the Bench4BL data directory to find bug and duplicate counts for each repository

In [14]:
project_data = []

# Walk through the directory structure: data/{group}/{project}
for group_name in os.listdir(bench4bl_directory):
    group_path = os.path.join(bench4bl_directory, group_name)
    if not os.path.isdir(group_path):
        continue
    # print(f"group path: {group_path}")

    for repo_name in os.listdir(group_path):
        repo_path = os.path.join(group_path, repo_name)
        if not os.path.isdir(repo_path):
            continue
        # print(f"repo path: {repo_path}")

        bugrepo_path = os.path.join(repo_path, 'bugrepo')
        if not os.path.isdir(bugrepo_path):
            continue

        # File paths to analyze
        xml_path = os.path.join(bugrepo_path, 'repository.xml')
        json_path = os.path.join(bugrepo_path, 'duplicates.json')

        total_bugs = 0
        duplicate_bugs = 0

        # 1. Count total bugs from the merged XML file
        if os.path.exists(xml_path):
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()
                # the number of <bug> tags in the merged file is the total count of unique bugs
                total_bugs = len(root.findall('bug'))
            except ET.ParseError:
                print(f"Warning: Could not parse XML for {repo_name}")

        # 2. Count duplicates from the duplicates.jso file
        if os.path.exists(json_path):
            # Check if the JSON file is empty before parsing
            if os.path.getsize(json_path) > 0:
                try: 
                    with open(json_path, 'r') as f:
                            # --- *** THE FIX IS HERE: Read and clean comments before parsing *** ---
                            # Read the raw content of the file
                            raw_content = f.read()
                            # Use a regular expression to remove comments (lines starting with # or parts of lines after #)
                            cleaned_content = re.sub(r'#.*', '', raw_content)
                            
                            # Parse the cleaned string
                            duplicates_data = json.loads(cleaned_content)
                            
                            # Handle both dict and list formats
                            if isinstance(duplicates_data, dict):
                                if duplicates_data:
                                    duplicate_list = list(duplicates_data.values())[0]
                                    duplicate_bugs = len(duplicate_list)
                            elif isinstance(duplicates_data, list):
                                duplicate_bugs = len(duplicates_data)
                except (json.JSONDecodeError, IndexError):
                    print(f"Warning: Could not parse non-empty JSON for {repo_name}")
        
        if total_bugs > 0 or duplicate_bugs > 0:
            project_data.append({
                'Group': group_name,
                'Repo_Name' : repo_name,
                'Total_Bug_Reports' : total_bugs,
                'Duplicate_Bug_Reports': duplicate_bugs
            })

if not project_data:
    print("No project data could be extracted. Please verify the folder structure and file contents.")


# Convert list of dicts to a DataFrame
results_df = pd.DataFrame(project_data)

# Sort the results first by Group, then by Total Bug Reports
sorted_results = results_df.sort_values(by=['Group', 'Total_Bug_Reports'], ascending=[True, False])
sorted_results.reset_index(drop=True, inplace=True)

print(" ---- Bench4BL analysis result ---")
print(sorted_results.to_string())


 ---- Bench4BL analysis result ---
       Group      Repo_Name  Total_Bug_Reports  Duplicate_Bug_Reports
0     Apache          CAMEL               1469                     50
1     Apache           HIVE               1241                    270
2     Apache          HBASE                838                     80
3    Commons           MATH                245                      8
4    Commons           LANG                217                     23
5    Commons  CONFIGURATION                133                      4
6    Commons       COMPRESS                113                      9
7    Commons    COLLECTIONS                 92                     16
8    Commons             IO                 91                      7
9    Commons          CODEC                 42                      2
10   Commons            CSV                 14                      0
11   Commons         CRYPTO                  8                      0
12   Commons         WEAVER                  2         

## Finding extensions of ground truth files from each repo

In [20]:
def get_file_extension(filepath):
    # Extracts the file extension from a given path.
    if '.' in os.path.basename(filepath):
        return os.path.splitext(filepath)[1]
    return 'no_extension'

In [21]:
def analyze_bench4bl_extensions(directory):
    '''
    Walks through the Bench4BL data directory to find all unique file extensions from the 
    'fixedFiles' sections of the repository.xml files.
    '''
    print("Analysing Fixed File Extensions in Bench4BL dataset")

    if not os.path.isdir(directory):
        print(f"Error: Directory not found at '{directory}'. Please check the path.")
        return
    
    all_fixed_files = []

    # Walk through the directory structure: data/{group}/{project}
    for group_name in os.listdir(directory):
        # skipping the Previous folder
        if group_name.lower() == 'previous':
            print(f"Skipping group: {group_name}")
            continue

        group_path = os.path.join(directory, group_name)
        if not os.path.isdir(group_path):
            continue

        for repo_name in os.listdir(group_path):
            repo_path = os.path.join(group_path, repo_name)
            if not os.path.isdir(repo_path):
                continue

            xml_path = os.path.join(repo_path, 'bugrepo', 'repository.xml')

            if os.path.exists(xml_path):
                try:
                    tree = ET.parse(xml_path)
                    root = tree.getroot()
                    # Find all <file> tags that are children of <fixedFiles>
                    file_nodes = root.findall('.//fixedFiles/file')
                    for node in file_nodes:
                        if node.text:
                            all_fixed_files.append(node.text.strip())
                except ET.ParseError:
                    print(f"Warning: Could not parse XML for {repo_name}")
    
    if not all_fixed_files:
        print("No fixed files could be extracted. Please verify the folder structure.")
        return
    
    # Get the unique extension for each file using a set
    unique_extensions = {get_file_extension(f) for f in all_fixed_files}

    print("Unique file extensions across all projects (exluding Previous)")
    print(sorted(list(unique_extensions)))

In [22]:
analyze_bench4bl_extensions(bench4bl_directory)

Analysing Fixed File Extensions in Bench4BL dataset
Skipping group: Previous
Unique file extensions across all projects (exluding Previous)
['.java']


# MetaData Extraction
Set of methods to calculate all ten metadata for selected projects from Bench4BL

1. LOC
2. age_years
3. median_bug_year
4. num_authors
5. num_commits
6. num_dependencies
7. polyglot_index
8. bug_density
9. code_complexity
10. bug_report_verbosity

In [15]:
# Import libraries
import pandas as pd
import os
import re
import git
import subprocess
from datetime import datetime
from tqdm import tqdm
import xml.etree.ElementTree as ET
import tempfile 

## Configuration

In [16]:
def collate_bench4BL_data(directory, mapping_csv_path):
    '''
    Walks through the Bench4BL data directory, parses all repository.xml files, and collates the bug report data into
    a single pandas Dataframe.
    '''
    print("Collating Bug Report from Bench4BL")

    if not os.path.isdir(directory):
        print(f"Error: Directory not found at '{directory}'. Please check the path.")
        return None
    
    # Load the manual mapping file
    try:
        mapping_df = pd.read_csv(mapping_csv_path)
        # Create a dictionary for easy lookup: {'folder_name': 'correct_repo_name'}
        repo_name_map = dict(zip(mapping_df['folder_name'], mapping_df['correct_repo_name']))
    except FileNotFoundError:
        print(f"Error: Mapping file not found at '{mapping_csv_path}")
        return None
    
    all_bugs_data = []

    # Walk through the directory structure: data/{group}/{project}
    for group_name in tqdm(os.listdir(directory), desc="Processing Groups"):
        # skip the 'Previous' folder
        if group_name.lower() == 'previous':
            continue

        group_path = os.path.join(directory, group_name)
        if not os.path.isdir(group_path):
            continue

        for repo_name in os.listdir(group_path):
            repo_path = os.path.join(group_path, repo_name)
            if not os.path.isdir(repo_path):
                continue

            xml_path = os.path.join(repo_path, 'bugrepo', 'repository.xml')

            if os.path.exists(xml_path):
                try:
                    tree = ET.parse(xml_path)
                    root = tree.getroot()

                    # Iterate through each <bug> tag in the XML
                    for bug in root.findall('bug'):
                        buginformation = bug.find('buginformation')
                        if buginformation is None:
                            continue

                        # Helper to safely get text from an element
                        def get_text(element, path):
                            node = element.find(path)
                            return node.text.strip() if node is not None and node.text is not None else ""
                        
                        # Extract fixed files into a list
                        fixed_files_nodes = bug.findall('.//fixedFiles/file')
                        fixed_files = [node.text.strip() for node in fixed_files_nodes if node.text]

                        # Use the mapping to get the correct repo name
                        correct_name = repo_name_map.get(repo_name, repo_name) # Fallback to folder name if not in map
                        repo = correct_name.split('/')[-1]
                        
                        # Create a dictionary for the current bug report
                        bug_data = {
                            'repo_name': correct_name,
                            'bug_id': f"{repo}-{bug.get('id')}", # Create a globally unique bug ID
                            'report_date': bug.get('opendate'),
                            'fix_date': bug.get('fixdate'),
                            'status': bug.get('resolution'),
                            'bug_report': get_text(buginformation, 'summary') + "\n" + get_text(buginformation, 'description'),
                            'version': get_text(buginformation, 'version'),
                            'fixed_version': get_text(buginformation, 'fixedVersion'),
                            'fixed_files': fixed_files
                        }
                        all_bugs_data.append(bug_data)
                    
                except ET.ParseError:
                    print(f"Warning: Could not parse XML for {repo_name}")
    
    if not all_bugs_data:
        print("No bug data could be extracted.")
        return None
    
    # Convert the list of dictionaries into a DataFrame
    final_df = pd.DataFrame(all_bugs_data)

    # Convert date columns to datetime objects for consistency
    final_df['report_date'] = pd.to_datetime(final_df['report_date'], errors='coerce')
    final_df['fix_date'] = pd.to_datetime(final_df['fix_date'], errors='coerce')

    return final_df
                        

In [17]:
def generate_repo_mapping_template(directory):
    """Generates a CSV template for manually correcting repo names."""
    print("--- Generating Repo Name Mapping Template ---")
    
    repo_names = []
    for group_name in os.listdir(directory):
        if group_name.lower() == 'previous':
            continue
        group_path = os.path.join(directory, group_name)
        if not os.path.isdir(group_path):
            continue
        for repo_name in os.listdir(group_path):
            repo_path = os.path.join(group_path, repo_name)
            if os.path.isdir(repo_path):
                repo_names.append({'folder_name': repo_name})

    if not repo_names:
        print("No repositories found to map.")
        return

    mapping_df = pd.DataFrame(repo_names).drop_duplicates().sort_values(by='folder_name')
    
    # Add an empty column for you to fill in
    mapping_df['correct_repo_name'] = ''
    
    mapping_df.to_csv(MAPPING_CSV_PATH, index=False)
    print(f"Success! A template file has been created at: {MAPPING_CSV_PATH}")
    print("Please open this file and fill in the 'correct_repo_name' column.")


In [25]:
def find_before_fix_sha(bench4bl_df, clone_dir):
    '''
    Finds the appropriate before_fix_sha for each bug report in the Bench4BL dataset.
    '''
    print(" Finding 'before_fix_sha' for each bug report ...")

    # Initialize the new column with a placeholder
    bench4bl_df['before_fix_sha'] = None
    bench4bl_df['language'] = 'java'

    # Group by repository to process one repo at a time for efficiency
    grouped = bench4bl_df.groupby('repo_name')

    for repo_name, group in tqdm(grouped, desc="Processing Repos"):
        # The 'language' isn't explicitly in Bench4BL, so we added the language 'Java; as Bench4BL contains only Java repos
        # This assume a flat structure in CLONE_DIR  -> CLONE_DIR/{language}/{repo_name} -? repo_name= apache/camel -> apache_camel
        language = 'java'
        repo_path = os.path.join(clone_dir, language, repo_name.replace('/', '_'))

        if not os.path.exists(repo_path):
            tqdm.write(f"Warning: Cloned repo not found for {repo_name}. Skipping.")
            continue

        try:
            repo = git.Repo(repo_path)

            # Get all commits and sort them by date (newest first)
            # This is a potentially slow, one-time operation per repo
            commits = sorted(list(repo.iter_commits()), key=lambda c: c.committed_datetime, reverse=True)

            if not commits:
                tqdm.write(f"Warning: No commits found for {repo_name}. Skipping...")
                continue

            # For each bug in this repo's group, find the right commit
            for index, bug_report in group.iterrows():
                report_date = bug_report['report_date']

                # Find the first commit whose date is earlier than or equal to the bug report date
                found_commit = None
                for commit in commits:
                    # GitPython datetime objects are timezone-aware, make the report_date aware too
                    if commit.committed_datetime <= report_date.tz_localize('UTC'):
                        found_commit = commit
                        break # Stop at the first (most recent match)

                if found_commit:
                    # Assign the commit hash to the correct row in the main DataFrame
                    bench4bl_df.loc[index, 'before_fix_sha'] = found_commit.hexsha

        except Exception as e:
            tqdm.write(f"An error occurred while processing {repo_name}: {e}")

    return bench4bl_df    


In [26]:
# 1. Path to the input CSV file
# Must have columns: 'repo_name', 'language', 'total_unique_bug_report'
INPUT_CSV_PATH = "/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_metadata_input.csv"

# 2. Path to the root directory of the Bench4BL dataset
BENCH4BL_DIRECTORY = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/Resources/Bench4BL/data/'

# Script to generate mapping for repo
MAPPING_CSV_PATH = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_repo_mapping.csv'
# generate_repo_mapping_template(BENCH4BL_DIRECTORY)

# load Bench4BL dataset in a pandas Dataframe
bench4bl_df = collate_bench4BL_data(BENCH4BL_DIRECTORY, MAPPING_CSV_PATH)
# Ensure report_date is a datetime object
bench4bl_df['report_date'] = pd.to_datetime(bench4bl_df['report_date'])

# 3. Path to the parent directory where repos are cloned
CLONE_DIR = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/temp_repos/'

# Run the function to add the Before fix commit SHAs in the dataset
# Note: currently only for cloned_repos we will find the before_fix_sha, later I will do it for all 
bench4bl_df = find_before_fix_sha(bench4bl_df, CLONE_DIR)

# Save the whole bench4bl dataset
bench4bl_df.to_csv('/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_full_data.csv')
print(f" Whole Bench4BL dataset collection and saved in a CSV file....")


# 4. Path for the final output CqSV file.
OUTPUT_CSV_PATH = '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_metadata.csv'

# 5. Extensions for Polyglot Index Calculation
LANGUAGE_EXTENSIONS = {
    'c++': ['.c', '.cc', '.cmake', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx', '.in', '.json', '.make', '.py', '.sh', '.xml'],
    'go': ['.go', '.json', '.proto', '.sh', '.yaml', '.yml'],
    'java': ['.gradle', '.groovy', '.java', '.json', '.properties', '.xml', '.yml', '.yaml'],
    'javascript': ['.css', '.html', '.js', '.json', '.jsx', '.mjs', '.scss', '.sh', '.ts', '.tsx', '.yaml', '.yml'],
    'kotlin': ['.gradle', '.json', '.kt', '.kts', '.properties', '.xml', '.yaml', '.yml'],
    'python': ['.bash', '.cfg', '.in', '.ini', '.json', '.py', '.sh', '.toml', '.yaml', '.yml']
}
PRIMARY_EXTENSIONS = {
    'python': ['.py'],
    'java': ['.java'],
    'kotlin': ['.kt'],
    'c++': ['.c', '.cc', '.cpp', '.cxx', '.h', '.hh', '.hpp', '.hxx'],
    'go': ['.go'],
    'javascript': ['.js', '.jsx', '.mjs', '.ts', '.tsx']
}

Collating Bug Report from Bench4BL


Processing Groups: 100%|██████████| 6/6 [00:00<00:00, 13.46it/s]


 Finding 'before_fix_sha' for each bug report ...


Processing Repos:   2%|▏         | 1/46 [09:20<7:00:36, 560.81s/it]

Processing Repos:  11%|█         | 5/46 [09:27<53:19, 78.03s/it]   

...

Processing Repos:  72%|███████▏  | 33/46 [14:07<01:54,  8.82s/it]

Processing Repos:  87%|████████▋ | 40/46 [14:10<00:21,  3.60s/it]

Processing Repos:  93%|█████████▎| 43/46 [16:37<00:57, 19.16s/it]

Processing Repos: 100%|██████████| 46/46 [17:04<00:00, 22.27s/it]


 Whole Bench4BL dataset collection and saved in a CSV file....


In [27]:
bench4bl_df.head()

,repo_name,bug_id,report_date,fix_date,status,bug_report,version,fixed_version,fixed_files,before_fix_sha,language
0,apache/camel,camel-72,2007-07-08 10:33:06,2007-07-09 09:00:19,Fixed,FileConfigureTest can&apos;t pass in Windows b...,1.1.0,1.1.0,[org.apache.camel.component.file.FileConfigure...,197253784368c28862c9b42d6e3044f2652817af,java
1,apache/camel,camel-81,2007-07-27 14:31:38,2007-07-30 16:49:10,Fixed,Stop logic a bit off in ServiceSupport.java\nW...,1.1.0,1.1.0,[org.apache.camel.impl.ServiceSupport.java],267d83aaafc317dc006ee59f051812b45c85173a,java
2,apache/camel,camel-85,2007-08-03 04:26:19,2007-08-03 20:18:56,Fixed,VM Component should extend Seda not Queue\nIt ...,1.1.0,1.1.0,[org.apache.camel.component.vm.VmComponent.java],d624cfe817a7ba5cb9da779e5797553e82cace27,java
3,apache/camel,camel-105,2007-08-14 21:27:14,2007-08-14 21:43:05,Fixed,FileProducer truncates message bodies > 256KB\...,1.1.0,1.2.0,[org.apache.camel.component.file.FileProducer....,b7891536ba89d6105682aab7d72827567e3f3275,java
4,apache/camel,camel-103,2007-08-14 18:22:55,2007-08-17 04:42:11,Fixed,ClassCastException when using GenericApplicati...,1.1.0,1.2.0,[org.apache.camel.spring.CamelContextFactoryBe...,471f8a7093c07f6e65bdcb01f4131bb06d74cb58,java


In [36]:
print(bench4bl_df['fixed_files'].apply(type).value_counts())

fixed_files
<class 'list'>    9461
Name: count, dtype: int64


## Helper Methods

In [28]:
# Pre-requisite: Assume 'bench4bl_df' is loaded and has the 'report_date' column
print(" Verifying the corrected 'report_date' column")

# The 'report_date' column should already be in datetime formate for your pre-processing
# we will just ensure it, coercing any potential errors that might still exist
bench4bl_df['report_date'] = pd.to_datetime(bench4bl_df['report_date'])

# 1. Show basic statistics of the corrected dates
print("Overall corrected date statistics:")
# Filter out any NaT values for accurate stats
valid_dates = bench4bl_df['report_date'].dropna()
if not valid_dates.empty:
    print(f" Earliest Date: {valid_dates.min()}")
    print(f" Latest Date: {valid_dates.max()}")
    print(f" Median Date: {valid_dates.median()}")
else:
    print(" No valid dates found in the 'report_date' column.")

# 2. Show the distribution of bug reports by year
print("Distribution of Bug Reports per Year (top 15):")
print(valid_dates.dt.year.value_counts().sort_index(ascending=False).head(15).to_string())

# 3. Isolate and check if any '1970' dates remain
print("Checking for any remaining anomalous '1970' dates")
problematic_rows = bench4bl_df[bench4bl_df['report_date'].dt.year == 1970]

if not problematic_rows.empty:
    print(f"Warning: Found {len(problematic_rows)} entries that still have a '1970' year")
else:
    print("Success: No entries with a '1970' year were found in the 'report_date' column.")

 Verifying the corrected 'report_date' column
Overall corrected date statistics:
 Earliest Date: 2004-12-18 09:09:13
 Latest Date: 2016-12-21 13:07:49
 Median Date: 2013-06-28 05:13:51
Distribution of Bug Reports per Year (top 15):
report_date
2016    1510
2015    1543
2014    1179
2013     765
2012     596
2011     816
2010     906
2009     959
2008     803
2007     222
2006     136
2005      25
2004       1
Checking for any remaining anomalous '1970' dates
Success: No entries with a '1970' year were found in the 'report_date' column.


In [29]:
def count_lines_in_file(file_path):
    '''
    Counts lines in a file, handling encoding errors.
    '''
    try:
        with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
            return len(f.readlines())
    except Exception:
        return 0

In [30]:
def calculate_loc_and_polyglot(repo_path, declared_language):
    '''
    Calculates LoC and polyglot index. Auto-detects the actual primary language in the snapshot to handle
    migrations
    '''
    loc_by_lang = {lang: 0 for lang in PRIMARY_EXTENSIONS.keys()}
    loc_relevant = 0

    # first, calculate LoC for each potential primary language
    for root, _, files in os.walk(repo_path):
        for file in files:
            for lang, exts in PRIMARY_EXTENSIONS.items():
                if file.endswith(tuple(exts)):
                    loc_by_lang[lang] += count_lines_in_file(os.path.join(root, file))

    # Auto-detect the language with the most LoC in this snapshot
    actual_primary_languge = max(loc_by_lang, key=loc_by_lang.get) if loc_by_lang else declared_language
    loc_primary = loc_by_lang.get(actual_primary_languge, 0)

    # Now, calculate total relevant LoC based on the DECLARED ecosystem
    relevant_exts = tuple(LANGUAGE_EXTENSIONS.get(declared_language.lower(), []))

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file.endswith(relevant_exts):
                loc_relevant += count_lines_in_file(os.path.join(root, file))
                
    polyglot_index = (loc_primary / loc_relevant) if loc_relevant > 0 else 0
    return loc_relevant, polyglot_index

In [31]:
def count_dependencies(repo_path, language):
    """
    Calculates a proxy for dependency complexity by summing the Lines of Code (LoC)
    of standard dependency files for the primary language.
    """
    loc_count = 0
    lang = language.lower()

    dependency_files = []
    if lang == 'python':
        dependency_files = ['requirements.txt', 'pyproject.toml']
    elif lang in ['java', 'kotlin']:
        dependency_files = ['pom.xml', 'build.gradle', 'build.gradle.kts']
    elif lang == 'c++':
        dependency_files = ['CMakeLists.txt', 'Makefile'] # Example for C++
    elif lang == 'javascript':
        dependency_files = ['package.json'] # Example for JS
    elif lang == 'go':
        dependency_files = ['go.mod'] # Example for Go

    for root, _, files in os.walk(repo_path):
        for file in files:
            if file in dependency_files:
                file_path = os.path.join(root, file)
                # Simply add the number of lines in the file to the count
                loc_count += count_lines_in_file(file_path)
    
    return loc_count

In [32]:
def calculate_average_complexity(repo_path, language):
    '''
    Calculates average cyclomatic complexity using the 'lizard' tool.
    We store the ouput of lizard to a temporary file to handle large outputs reliably.
    '''
    # Create a temp file to stroe the lizard output
    with tempfile.NamedTemporaryFile(mode='w+', delete=False, suffix='.txt', encoding='utf-8') as temp_out:
        temp_filename = temp_out.name

    try:
        # lizard expects 'c++' to be written as 'cpp'
        lang_for_lizard = 'cpp' if language.lower() == 'c++' else language.lower()
        # print("Language for lizard: ", lang_for_lizard)

        # Redirect stdout to the temp file
        # We run the command and tell it to write its output directly to our temp file
        result = subprocess.run(
            ['lizard', '-i', '0', repo_path], # another command ['lizard', '-l', 'lang_for_lizard', repo_path]
            stdout = open(temp_filename, 'w',  encoding='utf-8'), # Write stdout to the temp file
            stderr=subprocess.PIPE, # Still capture any errors in memory 
            check=False, text=True
        )

        # We can still manually check the return code if we want to log detailed errors
        if result.returncode != 0:
            print(f"\nWarning: Lizard finished with a non-zero exit code ({result.returncode}) for {repo_path}. This usually indicates warnings were found. Continuing to parse output.")
            # We don't return here, because the output file is likely still valid.

        # Read the results back from the file
        with open(temp_filename, 'r', encoding='utf-8') as f:
            lizard_output = f.read()

        # Get all non-empty lines from the output
        lines = [line for line in lizard_output.strip().splitlines()]

        # the summary data is on the second to last line
        if len(lines) >=3 :
            # target the line with the numbers (the last non-empty line)
            summary_line = lines[-1]

            # Split the line by whitespace
            values = summary_line.split()

            if len(values) >= 3:
                # The Avg CCN is the 3rd value (index 2)
                avg_ccn = float(values[2])
                return avg_ccn

        # Find the summary line in the output
        # If we reach here, the summary line was not found or was malformed
        print(f"\nWarning: Could not parse lizard summary for {repo_path}.")
        return 0.0
        
    except FileNotFoundError:
        # This error is critical, so we print it once and then it will return 0 for others.
        print("\nERROR: 'lizard' command not found. Please install it with 'pip install lizard'.")
        return 0.0
    except subprocess.CalledProcessError as e:
        print(f"\n Lizard command failed for {repo_path}. Stderr: {e.stderr}")
        return 0.0
    except (IndexError, ValueError) as e:
        print(f"\nFailed to extract complexity value from summary line for {repo_path}. Error: {e}")
        return 0.0
    except Exception as e:
        print(f"\nAn unexpected error occurred in calculate_average_complexity: {e}")
        return 0.0
    finally:
        if os.path.exists(temp_filename):
            os.remove(temp_filename)

## Main Code

In [33]:
print("Starting metadata extraction for Bench4BL Dataset...")

# Load input files
try:
    selected_repos_df = pd.read_csv(INPUT_CSV_PATH)
    # print(selected_repos_df)
except FileNotFoundError as e:
    print(f"Error: Input file not found. {e}")
    exit(0)

results = []

for _, row in tqdm(selected_repos_df.iterrows(), total=len(selected_repos_df), desc="Processing Repos"):
    repo_name = row['repo_name']
    language = row['language']
    unique_bugs = row['total_unique_bug_report']

    repo_path = os.path.join(CLONE_DIR, language.lower(), repo_name.replace('/', '_'))
    print(f"Calculating meta data for repo: {repo_name}")
    if not os.path.exists(repo_path):
        print(f"Warning: Clone repo not found for {repo_name} at {repo_path}. Skipping.")
        continue

    repo_bettlebox_data = bench4bl_df[bench4bl_df['repo_name'] == repo_name].copy()
    if repo_bettlebox_data.empty:
        print(f"Warning: No data found for {repo_name} in LCA main file. Skipping.")
        continue

    # Get the snapshot commit from the latest bug report
    latest_bug = repo_bettlebox_data.sort_values(by='report_date', ascending=False).iloc[0]
    snapshot_commit = latest_bug['before_fix_sha']

    try:
        repo = git.Repo(repo_path)
        repo.git.checkout(snapshot_commit, f=True)

        # *****************
        # Calculate Metrics
        # *****************

        # c) Age
        # --- Metrics now use the reliable 'report_date' column ---
        min_date = repo_bettlebox_data['report_date'].min()
        max_date = repo_bettlebox_data['report_date'].max()
        age_years = ((max_date - min_date).days) / 365.25
        median_bug_year = repo_bettlebox_data['report_date'].dt.year.median()

        # d) No. of authors & e) No. of commits (Correctly scoped to the snapshot)
        all_commits = list(repo.iter_commits())
        num_commits = len(all_commits)
        num_authors = len({c.author.email for c in all_commits})

        # a) LoC & g) Polyglot Index
        loc, polyglot_index = calculate_loc_and_polyglot(repo_path, language)

        # f) No. of external dependencies
        dependencies = count_dependencies(repo_path, language)

        # h) Bug density
        kloc = loc / 1000
        bug_density = (kloc / unique_bugs) if unique_bugs > 0 else 0

        # i) code_complexity
        code_complexity = calculate_average_complexity(repo_path, language)

        # j) Calculate bug report verbosoty
        bug_report_verbosity = repo_bettlebox_data['bug_report'].str.split().str.len().mean()
        
        results.append({
            'repo_name': repo_name,
            'language': language,
            'LoC': loc,
            'age_years': age_years,
            'median_bug_year': median_bug_year, # Raw data for later categorization
            'num_authors': num_authors,
            'num_commits': num_commits,
            'num_dependencies': dependencies,
            'polyglot_index': polyglot_index,
            'bug_density': bug_density,
            'code_complexity': code_complexity,
            'bug_report_verbosity': bug_report_verbosity
        })
    
    # except git.exec.GitCommandError as e:
    #     print(f"Error processing Git Repo {repo_name}: {e}")
    except Exception as e:
        print(f"An unexpected error occurred for {repo_name}: {e}")

if not results:
    print("No results were generated.")
    exit(0)

# Create final DataFrame
final_df = pd.DataFrame(results)

# # b) Calculate project_size category
# final_df = final_df.groupby('language', group_keys=False).apply(categorize_by_tercile)

# Save to CSV
final_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Metdata extraction complete. Results saved to '{OUTPUT_CSV_PATH}'")


Starting metadata extraction for Bench4BL Dataset...


Processing Repos:   0%|          | 0/21 [00:00<?, ?it/s]

Calculating meta data for repo: apache/camel


Processing Repos:   5%|▍         | 1/21 [00:36<12:12, 36.62s/it]


Calculating meta data for repo: apache/commons-compress


...

Calculating meta data for repo: wildfly/wildfly


Processing Repos:  95%|█████████▌| 20/21 [02:51<00:07,  7.15s/it]


Calculating meta data for repo: wildfly/wildfly-core


Processing Repos: 100%|██████████| 21/21 [03:06<00:00,  8.89s/it]


Metdata extraction complete. Results saved to '/home/user/CS21D002_A_Eashaan_Rao/Research/PhD/Objective1/benchmark_dataset_analysis/bench4bl_metadata.csv'
